# A tour of Retentioneering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/retentioneering/retentioneering-tools/blob/master/notebooks/retentioneering_5_tour.ipynb)

[Retentioneering](https://github.com/retentioneering/retentioneering-tools) is an open-source Python library for understanding user behavior from event logs: find where users get stuck, compare the paths different groups take and discover behavior patterns.

This notebook is the runnable companion to the README: sections 1–6 run the same code as its use cases. It needs no data upload: it uses the bundled e-commerce dataset, six months of a synthetic consumer-electronics store with 600 users, ~3,600 sessions and ~31,000 events. Every chart is interactive: click nodes and edges, drag, and open the sidebar (the panel icon in the top-right corner) to change parameters live. Use Python 3.10–3.13.

[Installation help](https://retentioneering.com/docs/installation) · [Documentation](https://retentioneering.com/docs/) · [Path patterns](https://retentioneering.com/docs/path-patterns)

In [ ]:
%pip install -q retentioneering

In [ ]:
import os

os.environ["RETENTIONEERING_NO_TRACK"] = "1"

import retentioneering as rete

stream = rete.datasets.load_ecom()
print("Retentioneering", rete.__version__)
print(
    "Events:",
    len(stream.df),
    "| Users:",
    stream.df["user_id"].nunique(),
    "| Sessions:",
    stream.df["session_id"].nunique(),
)

## The big picture

A transition graph of all sessions, with the path to purchase highlighted. Click **Default** above the graph to see all transitions, and click any event to focus on its connections.

In [ ]:
path = [
    "catalog",
    "product_view",
    "add_to_cart",
    "cart",
    "shipping_details",
    "payment_details",
    "purchase",
]

# pin the path to purchase to hand-picked positions; every other event keeps its automatic place
positions = {
    "catalog": {"x": -64, "y": 227},
    "product_view": {"x": -263, "y": 15},
    "add_to_cart": {"x": -277, "y": -165},
    "cart": {"x": 35, "y": -245},
    "shipping_details": {"x": 223, "y": -204},
    "payment_details": {"x": 257, "y": 72},
    "purchase": {"x": 111, "y": 242},
}

stream.transition_graph(
    path_col="session_id",
    views=[
        {
            "name": "Path to purchase",
            "focus": {"type": "path", "nodes": path},
            "hiddenEvents": ["path_start", "path_end"],
            "nodePositions": positions,
            "viewport": "fit",
        }
    ],
    view="Path to purchase",
    sidebar_open=False,
)

## 1. A key metric dropped. Which behavior change drove it?

Purchase conversion dropped between May 19 and June 7. Create a binary segment that [splits the dataset into two parts by date](https://retentioneering.com/docs/data-processors/add-segment#time_range--inside-vs-outside-a-window) – `inside` and `outside` that window – then compare behavior across them with the [transition graph](https://retentioneering.com/docs/widgets/transition-graph#diff-mode):

In [ ]:
periods = stream.add_segment("period", time_range=("2024-05-19", "2024-06-07"))
periods.transition_graph(diff=("period", "inside", "outside"))

Click the `payment_details` node to inspect the routes toward `purchase`, `payment_error` and `support_chat`. With this comparison order, red means a higher next-step probability inside the window; blue means lower.

Inside the window, the step from `payment_details` to `purchase` is 13 p.p. less likely, while `payment_details → payment_error` rises from 0.5% to 12%: the drop happens at the payment step. Use comparable observation windows and check the counts behind a difference; a path comparison supports diagnosis, while a causal conclusion needs the corresponding study design.

The same diff mode works for any group comparison: mobile versus desktop, acquisition channels, or experiment groups.

## 2. Where do users leave the funnel and what do they do instead?

Count paths that complete the steps in order with the [funnel](https://retentioneering.com/docs/widgets/funnel):

In [ ]:
steps = ["cart", "shipping_details", "purchase"]
stream.funnel(steps=steps, path_col="session_id")

Then compare sessions that stopped at the shipping stage of this funnel with those that completed it, using the [Step Matrix](https://retentioneering.com/docs/widgets/step-matrix) in diff mode:

In [ ]:
(
    stream.add_segment("stage", funnel_events=steps, path_col="session_id").step_matrix(
        path_pattern="cart->.*->shipping_details",
        step_window=2,
        path_col="session_id",
        diff=("stage", "shipping_details", "purchase"),
    )
)

`path_pattern="cart->.*->shipping_details"` splits the matrix into two blocks, one around `cart` and one around `shipping_details`; whatever happened between them is folded into the gap. Compare sessions that reached `shipping_details` but not `purchase` with those that completed the funnel: hover a cell to see each group's values, and click the arrows next to a block's `0` column to sort rows by preceding or following steps.

## 3. What leads to, or follows, an error or another key event?

[Step Sankey](https://retentioneering.com/docs/widgets/step-sankey) shows the same numbers as the Step Matrix, drawn as flows. Align all sessions on `payment_error` and show the two steps on either side of it:

In [ ]:
stream.step_sankey(
    anchor="payment_error",
    step_window=2,
    path_col="session_id",
)

A plain event name anchors each path on its first occurrence. An [anchor spec](https://retentioneering.com/docs/data-processors/truncate-paths#anchoring-on-a-sequence-not-just-an-event) chooses the position more precisely: which occurrence to use, which event of a [pattern](https://retentioneering.com/docs/path-patterns) to center on (`at`), or how far to shift from it (`offset`). Align sessions on their *last* payment error to see whether users recover after it or give up:

In [ ]:
stream.step_sankey(
    anchor={"pattern": "payment_error", "occurrence": "last"},
    step_window=2,
    path_col="session_id",
)

After the last error, sessions end more often than after the first one: `path_end` takes 19% of the next step instead of 14%, and 35% two steps later instead of 26%.

## 4. Which sessions match a behavior I care about?

Path patterns are a small regex-like language over events: `.*` is any run of events, `[a|b]` is one of the listed events, `[^a|b]*` is a run with none of them, and `path_start` / `path_end` are the path's boundaries. Find sessions that opened the cart, then ended without reaching shipping or support – a silent cart abandonment:

In [ ]:
abandoned = stream.filter_paths(
    {
        "metric": "matches_pattern",
        "op": "=",
        "value": True,
        "metric_args": {"pattern": "cart->[^shipping_details|support_chat]*->path_end"},
    },
    path_col="session_id",
)
abandoned.step_sankey(path_pattern="cart", path_col="session_id", step_window=3)

[`filter_paths`](https://retentioneering.com/docs/data-processors/filter-paths) keeps paths by a [path metric](https://retentioneering.com/docs/path-metrics) value; here `matches_pattern` checks each session against the pattern, and the Sankey aligns the matching sessions on `cart`. How many sessions is that?

In [ ]:
print(
    f"{abandoned.df['session_id'].nunique()} of {stream.df['session_id'].nunique()} sessions match"
)

## 5. How does product usage differ between users? What behavioral patterns are represented?

Start with a bare call to [Cluster Analysis](https://retentioneering.com/docs/widgets/cluster-analysis) and set everything up in its sidebar: pick features and a cluster range, add overview metrics, then click **Apply**.

In [ ]:
stream.cluster_analysis()

Here users are clustered by [`event_count_bulk`](https://retentioneering.com/docs/path-metrics), which expands into one count per event type, over a grid of 3 to 8 clusters scored by the [silhouette metric](https://en.wikipedia.org/wiki/Silhouette_(clustering)). The heatmap compares mean metric values across clusters (blue for lower values, red for higher). Overview metrics such as `length` or `in_segment_bulk` describe the clusters without changing how they are built. Pick another partition on the **Silhouette** tab, rename clusters in the header, and **Save Clusters** adds the split as a [segment](https://retentioneering.com/docs/segments) and shows the code that reproduces it. The same configuration can also be passed directly:

In [ ]:
stream.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": "3-8"},
    overview_metrics=[
        {"metric": "event_count_bulk"},
        {"metric": "length"},
        {
            "metric": "in_segment_bulk",
            "metric_args": {"segment_name": "acquisition_channel"},
        },
    ],
)

And the partition saved in the recording – four clusters, renamed – becomes a segment column with [add_clusters](https://retentioneering.com/docs/data-processors/add-clusters) and [rename_segment_levels](https://retentioneering.com/docs/data-processors/rename-segment-levels):

In [ ]:
user_types = stream.add_clusters(
    "user_type",
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": 4},
).rename_segment_levels(
    "user_type",
    {
        "cluster_0": "browsers",
        "cluster_1": "researchers",
        "cluster_2": "buyers",
        "cluster_3": "light_users",
    },
)

Once clusters are saved as a segment, explore them like any other segment: compare them in the [diff mode](https://retentioneering.com/docs/widgets#diff-mode) of any widget or side by side in [Segment Overview](https://retentioneering.com/docs/widgets/segment-overview).

Treat clusters as exploratory descriptions; check stability and usefulness before using them for targeting.

## 6. How does behavior change across visits?

[Cluster](https://retentioneering.com/docs/widgets/cluster-analysis) sessions by behavior type:

In [ ]:
stream.cluster_analysis(
    features=[{"metric": "event_count_bulk"}],
    method_args={"n_clusters": "4-8"},
    path_col="session_id",
)

Suppose 4 clusters look best. [`add_clusters`](https://retentioneering.com/docs/data-processors/add-clusters) reproduces that split as the `session_type` segment – the same result as **Save clusters** in the widget. Then [`collapse_events`](https://retentioneering.com/docs/data-processors/collapse-events) turns each session into a single event named by its type, so the graph shows how users move between kinds of visits:

In [ ]:
visits = (
    stream.add_clusters(
        "session_type",
        features=[{"metric": "event_count_bulk"}],
        method_args={"n_clusters": 4},
        path_col="session_id",
    )
    .collapse_events(group_col="session_id", name={"col": "session_type"})
    .rename_events(
        {
            "cluster_0": "browsing",
            "cluster_1": "quick_visit",
            "cluster_2": "purchase_visit",
            "cluster_3": "promo_visit",
        }
    )
)
visits.transition_graph()

## More to try

### Newcomers vs. regulars

Diff mode works with any segment column. Compare new and loyal users step by step from the start of a session:

In [ ]:
stream.step_matrix(
    diff=["user_lifecycle", "new", "loyal"],
    path_col="session_id",
    max_steps=8,
    sidebar_open=False,
)

### Cut paths around a pattern

Patterns also mark where to cut paths. Keep only the part of each session that starts at a failed payment and ends at the support chat that followed without a retry:

In [ ]:
after_error = stream.truncate_paths(
    start_anchor={"pattern": "payment_details->payment_error"},
    end_anchor={"pattern": "payment_error->[^payment_details]*->support_chat"},
    path_col="session_id",
)
after_error.to_dataframe().head(10)

### Zoom in on every funnel step

In Step Matrix and Step Sankey a pattern can set several anchors. Here each of the two funnel events gets its own two-step neighborhood, and whatever happened between them is folded into the gap:

In [ ]:
stream.step_sankey(
    path_pattern="shipping_details->.*->payment_details",
    step_window=2,
    path_col="session_id",
    sidebar_open=False,
)

## Check and share

Every widget has a headless `*_data()` twin that returns the numbers behind the chart. Inspect the table, then export the interactive view: the HTML opens without Python; recalculation needs the notebook.

In [ ]:
table = stream.transition_graph_data(path_col="session_id", edge_weight="count")
graph = stream.transition_graph(path_col="session_id")
graph.export_html("session-journeys.html")
table.head()

## Hand the analysis to an AI agent

Retentioneering ships a local MCP server (in beta). Start it from a notebook on your machine and connect Claude Code, Codex, Cursor or another MCP client:

```python
rete.mcp.serve(stream, context={"description": "E-commerce store. Main KPI: purchase conversion."})
```

The agent calls the library's tools and builds an interactive HTML report in which every number links to the chart it came from. The MCP server needs a local Python environment, so it doesn't run in Colab. See the [MCP server guide](https://retentioneering.com/docs/mcp-server).

## Try your own events

The minimum input is three columns: `user_id`, `event` and `timestamp`. For other column names, use a [schema](https://retentioneering.com/docs/eventstream#schema). A path can also be an agent run, a support conversation or another ordered process.

```python
import pandas as pd
events = pd.read_csv("events.csv", parse_dates=["timestamp"])
own_stream = rete.Eventstream(events)
own_stream.transition_graph()
```

Before interpreting your own results, inspect identities, duplicates, event order and the observation window. The bundled teaching data itself contains duplicate rows and timestamp ties; its planted patterns demonstrate the API, not causal evidence about a real business. See [path analysis](https://retentioneering.com/docs/path-analysis) for interpretation and [the dataset discussion](https://github.com/retentioneering/retentioneering-tools/issues/88) for its ordering limitations.

Next: [analyze agent runs](https://colab.research.google.com/github/retentioneering/retentioneering-tools/blob/e95d44cda4cc5ac29f148b47470c31b08a240854/notebooks/agent_path_analysis.ipynb) · [more analysis recipes](https://retentioneering.com/docs/recipes) · [ask a question](https://github.com/retentioneering/retentioneering-tools/issues)